Error Mitigation | Week 7
## Density Matrices, Channels & CPTP Maps — Exercise Solutions

This notebook implements and verifies all four exercises from Lecture 0:
1. **Bloch Vector** — Validate a density matrix, extract its Bloch vector, check purity
2. **Partial Trace** — Reduced states for product vs entangled states
3. **Kraus Completeness** — Verify CPTP condition for the depolarising channel
4. **Channel Composition** — Cascaded phase-flip channels

**Bonus:** Comparison of all four standard noise channels on the Bloch sphere

---
## Setup: Pauli Matrices & Helper Functions

In [1]:
import numpy as np
from numpy.linalg import eigvalsh

# ── Pauli matrices ──
I = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)

# ── Basis kets ──
ket0 = np.array([[1], [0]], dtype=complex)
ket1 = np.array([[0], [1]], dtype=complex)
ket_plus  = (ket0 + ket1) / np.sqrt(2)
ket_minus = (ket0 - ket1) / np.sqrt(2)

print("Pauli matrices and basis kets loaded.")

Pauli matrices and basis kets loaded.


In [ ]:
# ── Helper functions ──

def outer(psi):
    """Return |ψ⟩⟨ψ|."""
    return psi @ psi.conj().T

def bloch_vector(rho):
    """Extract Bloch vector (rx, ry, rz) from a 2x2 density matrix."""
    rx = np.real(np.trace(X @ rho))
    ry = np.real(np.trace(Y @ rho))
    rz = np.real(np.trace(Z @ rho))
    return np.array([rx, ry, rz])

def purity(rho):
    """Compute Tr(rho^2)."""
    return np.real(np.trace(rho @ rho))

def partial_trace_B(rho_AB):
    """Partial trace over qubit B of a 4x4 two-qubit density matrix."""
    T = rho_AB.reshape(2, 2, 2, 2)
    return np.einsum('ibjb->ij', T)

def phase_flip(rho, p):
    """Phase-flip channel: E(rho) = (1-p)rho + p Z rho Z."""
    return (1 - p) * rho + p * Z @ rho @ Z

def depolarising(rho, p):
    """Depolarising channel: E(rho) = (1-p)rho + p*I/2."""
    return (1 - p) * rho + p * I / 2

def amplitude_damping(rho, gamma):
    """Amplitude-damping channel with parameter gamma."""
    K0 = np.array([[1, 0], [0, np.sqrt(1 - gamma)]], dtype=complex)
    K1 = np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)
    return K0 @ rho @ K0.conj().T + K1 @ rho @ K1.conj().T

print("Helper functions defined.")

---
## Exercise 1: Bloch Vector

**Task:** Show that $\hat{\rho} = \frac{1}{2}\begin{pmatrix} 1.6 & 0.3 \\ 0.3 & 0.4 \end{pmatrix}$ is a valid density matrix, find its Bloch vector, and decide whether it is pure.

In [ ]:
rho = 0.5 * np.array([[1.6, 0.3],
                       [0.3, 0.4]], dtype=complex)

print("rho =")
print(rho)
print()

In [ ]:
# Step 1: Verify the three defining properties

# 1. Hermiticity
is_hermitian = np.allclose(rho, rho.conj().T)
print(f"1. Hermitian (rho = rho†):  {is_hermitian}")

# 2. Unit trace
tr = np.real(np.trace(rho))
print(f"2. Tr(rho) = {tr}")

# 3. Positive semidefinite
eigs = eigvalsh(rho)
is_psd = np.all(eigs >= -1e-12)
print(f"3. Eigenvalues: {np.round(eigs, 6)}  (all >= 0: {is_psd})")

print(f"\n=> Valid density matrix: {is_hermitian and np.isclose(tr, 1.0) and is_psd}")

In [ ]:
# Step 2: Find the Bloch vector

r = bloch_vector(rho)
r_norm = np.linalg.norm(r)

print(f"Bloch vector r = ({r[0]:.4f}, {r[1]:.4f}, {r[2]:.4f})")
print(f"|r| = {r_norm:.6f}")

In [ ]:
# Step 3: Pure or mixed?

g = purity(rho)
expected = (1 + r_norm**2) / 2

print(f"Purity Tr(rho^2) = {g:.6f}")
print(f"(1 + |r|^2)/2   = {expected:.6f}  [cross-check: match = {np.isclose(g, expected)}]")
print(f"\nPure? {'Yes' if np.isclose(r_norm, 1.0) else 'No'}"
      f" — |r| = {r_norm:.4f} < 1  =>  MIXED STATE")

---
## Exercise 2: Partial Trace

**Task:** Compute $\hat{\rho}_A$ for the product state $|+\rangle \otimes |0\rangle$ and for $|\Psi^-\rangle = \frac{1}{\sqrt{2}}(|01\rangle - |10\rangle)$. Which is pure?

In [ ]:
# (a) Product state |+⟩ ⊗ |0⟩

psi_a = np.kron(ket_plus, ket0)
rho_A_a = partial_trace_B(outer(psi_a))
g_a = purity(rho_A_a)

print("(a) |+⟩ ⊗ |0⟩  (product state)")
print(f"    rho_A =")
print(f"    {np.round(rho_A_a, 4)}")
print(f"    Purity = {g_a:.4f}  =>  {'PURE' if np.isclose(g_a, 1) else 'Mixed'}")
print(f"    Equals |+⟩⟨+|?  {np.allclose(rho_A_a, outer(ket_plus))}")

In [ ]:
# (b) Singlet state |Ψ⁻⟩ = (|01⟩ − |10⟩)/√2

psi_b = (np.kron(ket0, ket1) - np.kron(ket1, ket0)) / np.sqrt(2)
rho_A_b = partial_trace_B(outer(psi_b))
g_b = purity(rho_A_b)

print("(b) |Ψ⁻⟩ = (|01⟩ − |10⟩)/√2  (entangled)")
print(f"    rho_A =")
print(f"    {np.round(rho_A_b, 4)}")
print(f"    Purity = {g_b:.4f}  =>  {'Pure' if np.isclose(g_b, 1) else 'MAXIMALLY MIXED'}")
print(f"    Equals I/2?  {np.allclose(rho_A_b, I / 2)}")

**Conclusion:**
- Product state $|+\rangle\otimes|0\rangle$ → $\hat{\rho}_A = |+\rangle\langle+|$ is **pure** (no entanglement, no information loss)
- Singlet $|\Psi^-\rangle$ → $\hat{\rho}_A = I/2$ is **maximally mixed** (entanglement has transferred all local coherences into correlations with $B$)

---
## Exercise 3: Kraus Completeness — Depolarising Channel

**Task:** Verify $\sum_k K_k^\dagger K_k = I$ for the Kraus set $\{\sqrt{1 - 3p/4}\, I,\; \sqrt{p/4}\, X,\; \sqrt{p/4}\, Y,\; \sqrt{p/4}\, Z\}$.

In [ ]:
p = 0.1  # arbitrary test value — result holds for any p in [0,1]

K0 = np.sqrt(1 - 3*p/4) * I
K1 = np.sqrt(p/4) * X
K2 = np.sqrt(p/4) * Y
K3 = np.sqrt(p/4) * Z

kraus_ops = [K0, K1, K2, K3]
labels = ["√(1−3p/4)·I", "√(p/4)·X", "√(p/4)·Y", "√(p/4)·Z"]

print(f"Using p = {p}\n")
print("Individual K†K terms:")

total = np.zeros((2, 2), dtype=complex)
for lbl, K in zip(labels, kraus_ops):
    KdK = K.conj().T @ K
    total += KdK
    coeff = np.real(KdK[0, 0])
    print(f"  K†K for {lbl:16s} = {coeff:.6f} · I")

print(f"\nΣ K†K =")
print(np.round(np.real(total), 10))
print(f"\nEquals I?  {np.allclose(total, I)}")
print(f"\nSymbolic: (1 − 3p/4) + 3·(p/4) = {(1 - 3*p/4) + 3*(p/4):.1f}  ✓")

In [ ]:
# Verify trace preservation on random density matrices

print("Trace preservation on random states:")
np.random.seed(42)
for trial in range(5):
    A = np.random.randn(2, 2) + 1j * np.random.randn(2, 2)
    rho_rand = A @ A.conj().T
    rho_rand /= np.trace(rho_rand)
    
    rho_out = sum(K @ rho_rand @ K.conj().T for K in kraus_ops)
    print(f"  Trial {trial+1}: Tr(input) = {np.real(np.trace(rho_rand)):.6f}, "
          f"Tr(output) = {np.real(np.trace(rho_out)):.6f}")

---
## Exercise 4: Composition of Two Phase-Flip Channels

**Task:** Apply a phase-flip channel with $p_1$ followed by one with $p_2$. Show the result is a phase-flip channel with $1 - 2p_{\text{eff}} = (1 - 2p_1)(1 - 2p_2)$, and hence for small $p$, $p_{\text{eff}} \approx p_1 + p_2$.

In [ ]:
p1, p2 = 0.1, 0.15

# Analytical prediction
p_eff = p1 + p2 - 2*p1*p2
factor = (1 - 2*p1) * (1 - 2*p2)

print(f"p1 = {p1},  p2 = {p2}")
print(f"\np_eff = p1 + p2 − 2·p1·p2 = {p_eff:.6f}")
print(f"1 − 2·p_eff      = {1 - 2*p_eff:.6f}")
print(f"(1−2p1)(1−2p2)   = {factor:.6f}")
print(f"Match?  {np.isclose(1 - 2*p_eff, factor)}")

In [ ]:
# Numerical verification on |+⟩

rho_in  = outer(ket_plus)
rho_seq = phase_flip(phase_flip(rho_in, p1), p2)   # sequential
rho_eff = phase_flip(rho_in, p_eff)                 # single effective

print("Numerical verification on |+⟩:")
print(f"  Input  ρ₀₁          = {np.real(rho_in[0,1]):.6f}")
print(f"  After E_p2 ∘ E_p1   = {np.real(rho_seq[0,1]):.6f}")
print(f"  Single E_p_eff      = {np.real(rho_eff[0,1]):.6f}")
print(f"  Match?  {np.allclose(rho_seq, rho_eff)}")

In [ ]:
# Small-p approximation: p_eff ≈ p1 + p2

print(f"{'p1':>6s} {'p2':>6s} {'p_eff':>10s} {'p1+p2':>10s} {'error':>10s}")
print("-" * 46)
for pv in [0.01, 0.05, 0.1, 0.2, 0.3]:
    pe = pv + pv - 2*pv*pv
    pa = 2*pv
    print(f"{pv:6.2f} {pv:6.2f} {pe:10.6f} {pa:10.6f} {abs(pe-pa):10.6f}")

print("\nFor small p, the quadratic term 2·p1·p2 is negligible => p_eff ≈ p1 + p2  ✓")

---
## Bonus: Four Standard Channels Acting on |+⟩

Compare how bit-flip, phase-flip, depolarising, and amplitude-damping channels transform the Bloch vector and purity of $|+\rangle$ as the noise parameter sweeps from 0 to 0.5.

In [ ]:
rho_in = outer(ket_plus)

print(f"{'p/γ':>5s}  {'Channel':>15s}  {'rx':>7s} {'ry':>7s} {'rz':>7s}  {'|r|':>6s}  {'Pur.':>6s}")
print("-" * 62)

for param in [0.0, 0.1, 0.2, 0.3, 0.5]:
    channels = [
        ("Bit-flip",     (1-param)*rho_in + param*X@rho_in@X),
        ("Phase-flip",   phase_flip(rho_in, param)),
        ("Depolarising", depolarising(rho_in, param)),
        ("Amp. damping", amplitude_damping(rho_in, param)),
    ]
    for name, rho_out in channels:
        r = bloch_vector(rho_out)
        print(f"{param:5.2f}  {name:>15s}  {r[0]:+7.4f} {r[1]:+7.4f} {r[2]:+7.4f}"
              f"  {np.linalg.norm(r):6.4f}  {purity(rho_out):6.4f}")
    print()

### Observations

| Channel | Effect on $|+\rangle$ |
|:--|:--|
| **Bit-flip** | $|+\rangle$ is an eigenstate of $X$, so it's unaffected (rx stays 1) |
| **Phase-flip** | Coherences decay: rx shrinks by $(1-2p)$ per application |
| **Depolarising** | Uniform shrinkage: $\vec{r} \to (1-p)\vec{r}$ |
| **Amp. damping** | Asymmetric: rx shrinks by $\sqrt{1-\gamma}$, rz drifts toward +1 (north pole) |